## **Insurance Claims Aggregation (Window Functions) & Analytics Pipeline**

## **Parameter widgets for Orchestration job**

In [0]:
dbutils.widgets.text("run_date", "2026-02-19")
dbutils.widgets.text("env", "dev")
dbutils.widgets.text("threshold", "0.8")
dbutils.widgets.text("output_table", "insurance.insurance_agg_pipeline")

# Fetch parameter values
run_date = dbutils.widgets.get("run_date")
env = dbutils.widgets.get("env")
threshold = float(dbutils.widgets.get("threshold"))
output_table = dbutils.widgets.get("output_table")


### Notebook Purpose: This notebook loads curated Gold Zone data (insurance_gold) from Delta Lake, applies advanced aggregations and window functions, and persists results into the Reporting Zone for BI and analytics.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window 
# Load transformed Delta table 
df_agg = spark.table("insurance.insurance_gold")

In [0]:
#Agg Window Function 1 : Top Claim per Customer
Window_cust = Window.partitionBy("Customer_id").orderBy(F.desc("Claim_amount"))
df_top_claim = df_agg.withColumn("Rank", F.rank().over(Window_cust)) \
                     .filter(F.col("Rank") == 1)
df_top_claim.display()


Customer_id,Claim_amount,Claim_date,Claim_id,Claim_status,Claim_type,Policy_date,Premium,Fraud_flag,Claim_ratio,Risk_segment,claim_count,Claim_month,Claim_year,Rank
CU101,18500,2024-01-15,C001,Approved,Auto,2023-02-10,14500,1,1.2758620689655173,Medium Risk,null,1,2024,1
CU102,42000,2024-01-28,C002,Pending,Health,2023-03-05,22000,1,1.9090909090909092,Medium Risk,null,1,2024,1
CU103,9800,2024-02-10,C003,Rejected,Property,2023-04-12,12000,0,0.8166666666666667,Low Risk,null,2,2024,1
CU104,26500,2024-02-22,C004,Approved,Auto,2023-05-18,18000,1,1.4722222222222223,Medium Risk,null,2,2024,1
CU105,15000,2024-03-06,C005,Pending,Health,2023-06-09,16000,0,0.9375,Low Risk,null,3,2024,1
CU106,36500,2024-03-19,C006,Approved,Property,2023-07-21,24000,1,1.5208333333333333,Medium Risk,null,3,2024,1
CU107,7200,2024-04-03,C007,Rejected,Auto,2023-08-14,11000,0,0.6545454545454545,Low Risk,null,4,2024,1
CU108,49800,2024-04-18,C008,Approved,Health,2023-09-02,29000,1,1.717241379310345,Medium Risk,null,4,2024,1
CU109,31000,2024-05-01,C009,Pending,Property,2023-10-11,21000,1,1.4761904761904763,Medium Risk,null,5,2024,1
CU110,22500,2024-05-16,C010,Approved,Auto,2023-11-06,17000,1,1.3235294117647058,Medium Risk,null,5,2024,1


In [0]:
#Agg Window Function 2 :Running Total of Claims by Month
Window_month = Window.partitionBy("Claim_year").orderBy("Claim_month") \
                     .rowsBetween(Window.unboundedPreceding, Window.currentRow)
Monthly_trends = df_agg.groupBy("Claim_year", "Claim_month").agg(F.sum("Claim_amount").alias("Monthly_total")).orderBy(F.col("Claim_year").desc(),F.col("Claim_month").desc())
df_running_total = Monthly_trends.withColumn("Cumulative_claims", 
                                             F.sum("monthly_total").over(Window_month))
df_running_total.display()


Claim_year,Claim_month,Monthly_total,Cumulative_claims
2024,1,60500,60500
2024,2,36300,96800
2024,3,51500,148300
2024,4,57000,205300
2024,5,53500,258800
2024,6,54100,312900
2024,7,53000,365900
2024,8,34900,400800
2024,9,78500,479300
2024,10,56200,535500


In [0]:
#Agg Window Function 3 : Average Claim Amount per Customer (with window)
Window_avg = Window.partitionBy("Customer_id")
df_avg_claim = df_agg.withColumn("Avg_claim_per_customer", 
                                  F.avg("Claim_amount").over(Window_avg))
df_avg_claim.display()


Customer_id,Claim_amount,Claim_date,Claim_id,Claim_status,Claim_type,Policy_date,Premium,Fraud_flag,Claim_ratio,Risk_segment,claim_count,Claim_month,Claim_year,Avg_claim_per_customer
CU101,18500,2024-01-15,C001,Approved,Auto,2023-02-10,14500,1,1.2758620689655173,Medium Risk,null,1,2024,18500.0
CU102,42000,2024-01-28,C002,Pending,Health,2023-03-05,22000,1,1.9090909090909092,Medium Risk,null,1,2024,42000.0
CU103,9800,2024-02-10,C003,Rejected,Property,2023-04-12,12000,0,0.8166666666666667,Low Risk,null,2,2024,9800.0
CU104,26500,2024-02-22,C004,Approved,Auto,2023-05-18,18000,1,1.4722222222222223,Medium Risk,null,2,2024,26500.0
CU105,15000,2024-03-06,C005,Pending,Health,2023-06-09,16000,0,0.9375,Low Risk,null,3,2024,15000.0
CU106,36500,2024-03-19,C006,Approved,Property,2023-07-21,24000,1,1.5208333333333333,Medium Risk,null,3,2024,36500.0
CU107,7200,2024-04-03,C007,Rejected,Auto,2023-08-14,11000,0,0.6545454545454545,Low Risk,null,4,2024,7200.0
CU108,49800,2024-04-18,C008,Approved,Health,2023-09-02,29000,1,1.717241379310345,Medium Risk,null,4,2024,49800.0
CU109,31000,2024-05-01,C009,Pending,Property,2023-10-11,21000,1,1.4761904761904763,Medium Risk,null,5,2024,31000.0
CU110,22500,2024-05-16,C010,Approved,Auto,2023-11-06,17000,1,1.3235294117647058,Medium Risk,null,5,2024,22500.0


In [0]:
#Agg Window Function 4 : Rank Customers by Total Claim Amount
Window_rank = Window.orderBy(F.desc("Claim_amount"))
df_ranked = df_agg.withColumn("Global_rank", F.dense_rank().over(Window_rank))
df_ranked.display()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Customer_id,Claim_amount,Claim_date,Claim_id,Claim_status,Claim_type,Policy_date,Premium,Fraud_flag,Claim_ratio,Risk_segment,claim_count,Claim_month,Claim_year,Global_rank
CU180,50000,2025-10-25,C080,Approved,Health,2023-10-02,30000,1,1.6666666666666667,Medium Risk,null,10,2025,1
CU108,49800,2024-04-18,C008,Approved,Health,2023-09-02,29000,1,1.717241379310345,Medium Risk,null,4,2024,2
CU124,49500,2024-12-19,C024,Approved,Property,2024-09-02,29500,1,1.6779661016949152,Medium Risk,null,12,2024,3
CU162,49000,2025-11-29,C062,Approved,Health,2023-11-21,30000,1,1.6333333333333333,Medium Risk,null,11,2025,4
CU144,49000,2025-10-19,C044,Approved,Health,2024-09-01,30000,1,1.6333333333333333,Medium Risk,null,10,2025,4
CU132,48500,2025-04-18,C032,Approved,Health,2024-02-01,29000,1,1.6724137931034482,Medium Risk,null,4,2025,5
CU174,48000,2025-07-18,C074,Approved,Health,2023-04-04,29500,1,1.6271186440677967,Medium Risk,null,7,2025,6
CU168,47500,2025-12-26,C068,Approved,Health,2024-03-11,29000,1,1.6379310344827587,Medium Risk,null,12,2025,7
CU120,47000,2024-10-22,C020,Approved,Health,2024-06-20,30000,1,1.5666666666666667,Medium Risk,null,10,2024,8
CU156,47000,2025-06-30,C056,Approved,Health,2023-06-25,29500,1,1.5932203389830508,Medium Risk,null,6,2025,8


In [0]:
#Agg Window Function 5 : Year‑over‑Year Growth in Claims
Window_yoy = Window.orderBy("claim_year")
df_yoy = Monthly_trends.withColumn("Prev_year_total", 
                                   F.lag("Monthly_total").over(Window_yoy)) \
                       .withColumn("Yoy_growth", 
                                   (F.col("Monthly_total") - F.col("Prev_year_total")) / F.col("Prev_year_total"))
df_yoy.display()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Claim_year,Claim_month,Monthly_total,Prev_year_total,Yoy_growth
2024,1,60500,null,null
2024,2,36300,60500,-0.4
2024,3,51500,36300,0.418732782369146
2024,4,57000,51500,0.10679611650485436
2024,5,53500,57000,-0.06140350877192982
2024,6,54100,53500,0.011214953271028037
2024,7,53000,54100,-0.02033271719038817
2024,8,34900,53000,-0.34150943396226413
2024,9,78500,34900,1.2492836676217765
2024,10,56200,78500,-0.2840764331210191


In [0]:
df_agg= spark.table("insurance.insurance_gold")
df_agg.display()

Customer_id,Claim_amount,Claim_date,Claim_id,Claim_status,Claim_type,Policy_date,Premium,Fraud_flag,Claim_ratio,Risk_segment,claim_count,Claim_month,Claim_year
CU101,18500,2024-01-15,C001,Approved,Auto,2023-02-10,14500,1,1.2758620689655173,Medium Risk,null,1,2024
CU102,42000,2024-01-28,C002,Pending,Health,2023-03-05,22000,1,1.9090909090909092,Medium Risk,null,1,2024
CU103,9800,2024-02-10,C003,Rejected,Property,2023-04-12,12000,0,0.8166666666666667,Low Risk,null,2,2024
CU104,26500,2024-02-22,C004,Approved,Auto,2023-05-18,18000,1,1.4722222222222223,Medium Risk,null,2,2024
CU105,15000,2024-03-06,C005,Pending,Health,2023-06-09,16000,0,0.9375,Low Risk,null,3,2024
CU106,36500,2024-03-19,C006,Approved,Property,2023-07-21,24000,1,1.5208333333333333,Medium Risk,null,3,2024
CU107,7200,2024-04-03,C007,Rejected,Auto,2023-08-14,11000,0,0.6545454545454545,Low Risk,null,4,2024
CU108,49800,2024-04-18,C008,Approved,Health,2023-09-02,29000,1,1.717241379310345,Medium Risk,null,4,2024
CU109,31000,2024-05-01,C009,Pending,Property,2023-10-11,21000,1,1.4761904761904763,Medium Risk,null,5,2024
CU110,22500,2024-05-16,C010,Approved,Auto,2023-11-06,17000,1,1.3235294117647058,Medium Risk,null,5,2024


## **Final Delta Table Pipeline**

In [0]:
#Curated “Production‑ready” dataset
df_agg.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("insurance.insurance_pipeline")

In [0]:
#Save the most important ones into Delta tables so they can be queried by BI tools or reused later.
Monthly_trends.write.format("delta").mode("overwrite").saveAsTable("insurance.agg_monthly_trends")
df_ranked.write.format("delta").mode("overwrite").saveAsTable("insurance.agg_rank_total")
df_top_claim.write.format("delta").mode("overwrite").saveAsTable("insurance.agg_top_customers")
df_running_total.write.format("delta").mode("overwrite").saveAsTable("insurance.agg_running_total")
df_avg_claim.write.format("delta").mode("overwrite").saveAsTable("insurance.agg_claims_by_type")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Business Impact Summary

This aggregation pipeline enables insurers to:

- **Detect Fraud**: Fraud flags highlight claims exceeding premiums, reducing financial losses.
- **Segment Risk**: Claim ratios and risk tiers support premium pricing strategies.
- **Profile Customers**: Claim frequency and top claimants identify high-risk customers for monitoring.
- **Measure Profitability**: Loss ratio by claim type provides actuarial KPIs for business performance.
- **Forecast Trends**: Monthly and yearly claim totals support predictive analytics and resource planning.

By persisting these outputs into the Reporting Zone, the pipeline delivers curated insights that can be consumed by BI tools (Power BI, Tableau) for decision-making.
